Prepares raw data export datasets from Waters for downstream data analysis pipeline.
(1) merges target compound concentrations ('Calculated Concentration') from a no-NIS export into the main NIS-response file
(2) optionally swaps in calibration data from a separate file
(3) assigns Expected Concentration values for NIS, EIS, and special EIS compounds (HFPO-DA, N-MeFOSA, N-EtFOSA, FTCAs, FTUCAs, 10:2 FTS) based on sample type and injection name.
(4) Outputs a prepared_ CSV files that can be used as 'raw data' for create_project_folder.ipynb and data_analysis.ipynb

In [ ]:
import pandas as pd
from pathlib import Path

file_containing_nis_responses = r"paola/oysters/raw_data/20260814_Oysters_QuEChERS_Trial5.csv"
concentration_file_no_nis = r"paola/oysters/raw_data/20260814_Oysters_QuEChERS_Trial5_no_NIS.csv"
file_for_correct_calibration_data = r"simon/EPA_fish/260908_Lake_Trout/Lake_Trout_EIS/wet/20260902_EPA_PFAS_Lake_Trout_eis_recoveries_wet_core.csv"

nis_in_calibration = 1 #ng/sample
eis_in_calibration = 0.5 #ng/sample
eis_mefosa_etfosa_genx_in_calibration = 2.5 #ng/sample

nis_in_unknown = 4 #ng/sample
eis_in_unknown = 3 #ng/sample
eis_mefosa_etfosa_genx_in_unknown = 9 #ng/sample

target_keys = ["Injection Name", "Acquisition Date Time", "Compound Name"]

In [ ]:
df = pd.read_csv(file_containing_nis_responses)
#df = df[~df["Compound Name"].isin(["L-PFOS_2", "Br-PFOS_2"])].reset_index(drop=True)
concentrations_df = pd.read_csv(concentration_file_no_nis)

for _df in (df, concentrations_df):
    _df["Acquisition Date Time"] = pd.to_datetime(
        _df["Acquisition Date Time"], format="mixed", errors="raise"
    ).dt.strftime("%Y-%m-%d %H:%M")
    if file_for_correct_calibration_data:
        _df = _df[_df["Sample Type"] != "Standard"].reset_index(drop=True)

# target compound mask: not NIS, not EIS, not Avg
def is_target(name_series: pd.Series) -> pd.Series:
    return ~(
        name_series.str.contains("NIS", na=False)
        | name_series.str.contains("EIS", na=False)
        | name_series.str.contains("Avg", na=False)
    )

targets_main = df[is_target(df["Compound Name"])].copy()
targets_conc = concentrations_df[is_target(concentrations_df["Compound Name"])].copy()

# verify exact match on keys
keys_main = set(map(tuple, targets_main[target_keys].values))
keys_conc = set(map(tuple, targets_conc[target_keys].values))
missing_in_conc = keys_main - keys_conc
missing_in_main = keys_conc - keys_main
if missing_in_conc or missing_in_main:
    raise ValueError(
        f"Target compound row mismatch between files.\n"
        f"Missing in concentration file: {missing_in_conc}\n"
        f"Missing in file containing NIS responses: {missing_in_main}"
    )

# build lookup and overwrite Calculated Concentration
lookup = targets_conc.set_index(target_keys)["Calculated Concentration"]
target_mask = is_target(df["Compound Name"])
df.loc[target_mask, "Calculated Concentration"] = [
    lookup[(i, a, c)]
    for i, a, c in df.loc[target_mask, target_keys].itertuples(index=False, name=None)
]

In [ ]:

print("Before concat:", len(df), df['Sample Type'].value_counts(dropna=False).to_dict())
if file_for_correct_calibration_data is not None:
    new_std = pd.read_csv(file_for_correct_calibration_data)
    new_std["Acquisition Date Time"] = pd.to_datetime(
        new_std["Acquisition Date Time"], format="mixed", errors="raise"
    ).dt.strftime("%Y-%m-%d %H:%M")

    if set(df.columns) != set(new_std.columns):
        raise ValueError(f"Column mismatch: {set(df.columns) ^ set(new_std.columns)}")

    df = pd.concat([
        df[df['Sample Type'] != 'Standard'], new_std[new_std['Sample Type'] == 'Standard']
        ], ignore_index=True)

In [ ]:
special_eis = [
    'EIS-13C3_HFPO-DA', 'EIS-d3_N-MeFOSA', 'EIS-d5_N-EtFOSA', 'EIS-13C2_6:2 FTCA', 'EIS-13C2_8:2 FTCA',
    'EIS-13C2_10:2 FTCA', 'EIS-13C2_6:2 FTUCA', 'EIS-13C2_8:2 FTUCA', 'EIS-13C2_10:2 FTUCA', 'EIS-13C2_10:2 FTS',
    ]

for stypes, nis_val, eis_val, special_val in [
    (['Standard', 'Blank'], nis_in_calibration, eis_in_calibration, eis_mefosa_etfosa_genx_in_calibration),
    (['Unknown'], nis_in_unknown, eis_in_unknown, eis_mefosa_etfosa_genx_in_unknown),
]:
    mask_type = df['Sample Type'].isin(stypes)
    df.loc[mask_type & df['Compound Name'].str.contains('NIS', na=False), 'Expected Concentration'] = nis_val
    df.loc[mask_type & (df['Compound Name'].str.contains('EIS', na=False) | df['Compound Name'].str.contains('Avg', na=False)) & ~df['Compound Name'].isin(special_eis), 'Expected Concentration'] = eis_val
    df.loc[mask_type & df['Compound Name'].isin(special_eis), 'Expected Concentration'] = special_val

In [ ]:
mask = (df['Sample Type'] == 'Unknown') & (df['Injection Name'].isin(['PFAS CS6 1/5ng/mL', 'PFAS CS0 0/0ng/mL', 'PFAS CS0 0ng/mL']))
df.loc[mask & df['Compound Name'].str.contains('NIS', na=False), 'Expected Concentration'] = 1
df.loc[mask & (df['Compound Name'].str.contains('EIS', na=False) | df['Compound Name'].str.contains('Avg', na=False)) & ~df['Compound Name'].isin(special_eis), 'Expected Concentration'] = 0.5
df.loc[mask & df['Compound Name'].isin(special_eis), 'Expected Concentration'] = 2.5

mask = (df['Sample Type'] == 'Unknown') & (df['Injection Name'].isin(['PFAS CS6 4ng/mL']))
df.loc[mask & df['Compound Name'].str.contains('NIS', na=False), 'Expected Concentration'] = 4
df.loc[mask & (df['Compound Name'].str.contains('EIS', na=False) | df['Compound Name'].str.contains('Avg', na=False)) & ~df['Compound Name'].isin(special_eis), 'Expected Concentration'] = 4
df.loc[mask & df['Compound Name'].isin(special_eis), 'Expected Concentration'] = 12

In [ ]:
p = Path(file_containing_nis_responses)
df.to_csv(p.with_name(f'prepared_{p.name}'), index=False)

df['Sample Type'].unique()
print("After concat:", len(df), df['Sample Type'].value_counts(dropna=False).to_dict())